In [ ]:
import numpy as np
from numpy.random import rand, randint
import gurobipy as gp
from gurobipy import GRB

# https://docs.gurobi.com/

# --- API --- #
params = {
"WLSACCESSID": "bf9a9974-82e7-4745-8a9c-d853d1d65926",
"WLSSECRET": "a9bbbc67-d3f4-4958-8aa2-c2e5a7b0a357",
"LICENSEID": 2773076,
}
env = gp.Env(params=params)

# --- Setup --- #
scenes = ["Scene1", "Scene2", "Scene3", "Scene4", "Scene5", "Scene6", "Scene7", "Scene8", "Scene9"]
actors = ["Actor1", "Actor2", "Actor3", "Actor4", "Actor5", "Actor6", "Actor7", "Actor8", "Actor9"]
n = len(scenes)
m = len(actors)
holdingCost=[760, 620, 850, 920, 760, 720, 870, 650, 820]
durations = [50, 50, 30, 40, 30, 50, 20, 50, 40]
sceneDuration = dict(zip(scenes, durations))
actorHolding = dict(zip(actors, holdingCost))
W = 120

# Table 1
T =np.matrix([
    [1, 1, 0, 1, 1, 1, 0, 0, 0],
    [1, 1, 1, 0, 0, 1, 1, 1, 0],
    [0, 1, 0, 0, 0, 1, 1, 1, 1],
    [0, 1, 0, 0, 1, 1, 0, 1, 0],
    [0, 0, 1, 0, 1, 1, 0, 1, 0],
    [0, 1, 0, 1, 0, 0, 0, 1, 1],
    [0, 1, 1, 0, 0, 1, 1, 1, 0],
    [0, 0, 1, 1, 1, 0, 0, 1, 0],
    [0, 1, 1, 0, 0, 1, 1, 1, 0]
])

availability = {
    (w, s): T[j, i] for i, s in enumerate(scenes) for j, w in enumerate(actors) if T[j, i] == 1
}

# Table 2
S = np.matrix([
    [0, 2, 4, 3, 1, 1, 3, 4, 5],
    [9, 0, 9, 4, 6, 10, 7, 6, 6],
    [1, 6, 0, 5, 7, 1, 4, 6, 2],
    [4, 7, 5, 0, 1, 9, 6, 3, 5],
    [2, 4, 5, 2, 0, 1, 10, 5, 2],
    [2, 6, 8, 3, 2, 0, 1, 3, 10],
    [4, 9, 10, 5, 2, 1, 0, 8, 9],
    [10, 5, 8, 3, 2, 7, 6, 0, 10],
    [10, 6, 1, 7, 4, 1, 7, 5, 0]
])


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2773076
Academic license 2773076 - for non-commercial use only - registered to km___@uga.edu


In [ ]:
# --- Finding D and H --- #
# D: max number of days
# H: max number of scenes in a day

D = (2*n*(max(durations)+np.matrix.max(S)))/(W+np.matrix.max(S))
if np.ceil(D)%2 == 0:
  D = int(np.floor(D))
else:
  D=int(np.ceil(D))

increasingDurations = np.sort(durations)
increasingChangeover = np.array(np.sort(np.matrix.flatten(S)))[0]

vals = []
hs = []
for h in range(2, n+1):
  total = np.sum(increasingChangeover[:h]) + np.sum(increasingDurations[:h])
  if total <= W:
    # to make sure combination is within admissible set
    vals.append(total)
    hs.append(h)
H = hs[np.argmax(vals)]

In [5]:
# --- Finding subsets of scenes P --- #
#TODO: find decreasing subsets in relation of scene number

def finding_P(n, scenes, durations, W, idx, start):
    admissibleP = []

    # compute total duration of current subset
    total = sum(durations[i] for i in idx)

    # current subset is admissible
    admissibleP.append([scenes[i] for i in idx])

    for i in range(start, n):
        if total + durations[i] <= W:
            admissibleP += finding_P(n, scenes, durations, W, idx + [i], i+1)

    return admissibleP

P = finding_P(n, scenes, durations, W, [], 0)[1:]
k = len(P)
print(P)

[['Scene1'], ['Scene1', 'Scene2'], ['Scene1', 'Scene2', 'Scene7'], ['Scene1', 'Scene3'], ['Scene1', 'Scene3', 'Scene4'], ['Scene1', 'Scene3', 'Scene5'], ['Scene1', 'Scene3', 'Scene7'], ['Scene1', 'Scene3', 'Scene9'], ['Scene1', 'Scene4'], ['Scene1', 'Scene4', 'Scene5'], ['Scene1', 'Scene4', 'Scene7'], ['Scene1', 'Scene5'], ['Scene1', 'Scene5', 'Scene7'], ['Scene1', 'Scene5', 'Scene9'], ['Scene1', 'Scene6'], ['Scene1', 'Scene6', 'Scene7'], ['Scene1', 'Scene7'], ['Scene1', 'Scene7', 'Scene8'], ['Scene1', 'Scene7', 'Scene9'], ['Scene1', 'Scene8'], ['Scene1', 'Scene9'], ['Scene2'], ['Scene2', 'Scene3'], ['Scene2', 'Scene3', 'Scene4'], ['Scene2', 'Scene3', 'Scene5'], ['Scene2', 'Scene3', 'Scene7'], ['Scene2', 'Scene3', 'Scene9'], ['Scene2', 'Scene4'], ['Scene2', 'Scene4', 'Scene5'], ['Scene2', 'Scene4', 'Scene7'], ['Scene2', 'Scene5'], ['Scene2', 'Scene5', 'Scene7'], ['Scene2', 'Scene5', 'Scene9'], ['Scene2', 'Scene6'], ['Scene2', 'Scene6', 'Scene7'], ['Scene2', 'Scene7'], ['Scene2', 'Scene

In [6]:
# --- ILP_pat Model Setup --- #
ILP_pat = gp.Model("ILP_pat", env=env)

# decision variables
a = ILP_pat.addVars(k, n,vtype=GRB.BINARY, name="a")
x_pat = ILP_pat.addVars(k, D, vtype=GRB.BINARY, name="x_pat")
y_pat = ILP_pat.addVars(m, D, D, vtype=GRB.BINARY, name="y_pat")

# objective function
objFunc_pat = ILP_pat.setObjective(
    gp.quicksum(
        holdingCost[i] *
        gp.quicksum(
              (d2-d1+1)*y_pat[i, d1, d2]
              for d1 in range(D)
              for d2 in range(d1, D)
        )
        for i in range(len(actors))
    ), GRB.MINIMIZE
)

# --- constraints (lord help us) --- #
c21 = ILP_pat.addConstr(
    gp.quicksum(
        x_pat[k, 1] for k in range(len(P))
    ) == 1, name="c21"
) # Constraint (21)

c22 = ILP_pat.addConstrs((
    gp.quicksum(
        x_pat[k, d-1]
        for k in range(len(P))
        ) <=
    gp.quicksum(
        x_pat[k, d]
        for k in range(len(P))
        )
    for d in range(2, D)),
    name="c22"
) # Constraint (22)

c23 = ILP_pat.addConstrs((
    gp.quicksum(
        a[k, j] * x_pat[k, d]
        for k in range(len(P))
        for d in range(D)
    ) == 1
    for j in range(n)),
    name="c23"
) # Constraint (23)

c24 = ILP_pat.addConstrs((
    gp.quicksum(
        y_pat[i, d1, d2]
        for d1 in range(D)
        for d2 in range(d1, D)
    ) == 1
    for i in range(m)),
    name="c24"
) # Constraint (24)

c25 = ILP_pat.addConstrs((
    gp.quicksum(
        T[i, j] * a[k, j] * x_pat[k, d]
        for k in range(len(P))
        for j in range(n)
    ) <=
    gp.quicksum(
        y_pat[i, d1, d2]
        for d2 in range(d, D)
        for d1 in range(d)
    )
    for i in range(m) for d in range(D)),
    name="c25"
) # Constraint (25)

# --- Constraints (26-27) --- #
# These constraints are embedded in the init of the variables

In [7]:
# --- optimize that shit --- #
ILP_pat.optimize()
sol = ILP_pat.getJSONSolution()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: AMD Ryzen 7 4700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Academic license 2773076 - for non-commercial use only - registered to km___@uga.edu
Optimize a model with 17 rows, 2511 columns and 1890 nonzeros (Min)
Model fingerprint: 0x179b51ed
Model has 405 linear objective coefficients
Model has 90 quadratic constraints
Variable types: 0 continuous, 2511 integer (2511 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [6e+02, 8e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
  QRHS range       [1e+00, 1e+00]

Presolve added 21 rows and 0 columns
Presolve removed 0 rows and 382 columns
Presolve time: 0.12s
Presolved: 24176 rows, 10148 columns, 99365 nonzeros
Variable types: 0 contin

In [9]:
all_vars = ILP_pat.getVars()
values = ILP_pat.getAttr("X", all_vars)
names = ILP_pat.getAttr("VarName", all_vars)

for name, val in zip(names, values):
    if val == 1:
        print(f"{name} = {val}")

a[1,0] = 1.0
a[5,5] = 1.0
a[17,4] = 1.0
a[48,7] = 1.0
a[54,8] = 1.0
a[57,3] = 1.0
a[62,7] = 1.0
a[66,1] = 1.0
a[97,6] = 1.0
a[98,2] = 1.0
x_pat[1,7] = 1.0
x_pat[2,8] = 1.0
x_pat[4,8] = 1.0
x_pat[5,4] = 1.0
x_pat[7,8] = 1.0
x_pat[9,8] = 1.0
x_pat[13,8] = 1.0
x_pat[15,6] = 1.0
x_pat[15,8] = 1.0
x_pat[16,8] = 1.0
x_pat[17,3] = 1.0
x_pat[20,8] = 1.0
x_pat[21,5] = 1.0
x_pat[22,5] = 1.0
x_pat[22,8] = 1.0
x_pat[23,6] = 1.0
x_pat[24,8] = 1.0
x_pat[25,8] = 1.0
x_pat[27,8] = 1.0
x_pat[28,7] = 1.0
x_pat[30,1] = 1.0
x_pat[31,8] = 1.0
x_pat[32,8] = 1.0
x_pat[33,3] = 1.0
x_pat[33,8] = 1.0
x_pat[34,8] = 1.0
x_pat[35,4] = 1.0
x_pat[37,8] = 1.0
x_pat[38,6] = 1.0
x_pat[38,7] = 1.0
x_pat[38,8] = 1.0
x_pat[41,6] = 1.0
x_pat[41,7] = 1.0
x_pat[41,8] = 1.0
x_pat[42,8] = 1.0
x_pat[43,7] = 1.0
x_pat[44,8] = 1.0
x_pat[46,6] = 1.0
x_pat[46,8] = 1.0
x_pat[47,6] = 1.0
x_pat[49,8] = 1.0
x_pat[51,8] = 1.0
x_pat[52,8] = 1.0
x_pat[54,7] = 1.0
x_pat[57,8] = 1.0
x_pat[58,8] = 1.0
x_pat[60,8] = 1.0
x_pat[62,5] = 1.0
x_pa

In [10]:
P

[['Scene1'],
 ['Scene1', 'Scene2'],
 ['Scene1', 'Scene2', 'Scene7'],
 ['Scene1', 'Scene3'],
 ['Scene1', 'Scene3', 'Scene4'],
 ['Scene1', 'Scene3', 'Scene5'],
 ['Scene1', 'Scene3', 'Scene7'],
 ['Scene1', 'Scene3', 'Scene9'],
 ['Scene1', 'Scene4'],
 ['Scene1', 'Scene4', 'Scene5'],
 ['Scene1', 'Scene4', 'Scene7'],
 ['Scene1', 'Scene5'],
 ['Scene1', 'Scene5', 'Scene7'],
 ['Scene1', 'Scene5', 'Scene9'],
 ['Scene1', 'Scene6'],
 ['Scene1', 'Scene6', 'Scene7'],
 ['Scene1', 'Scene7'],
 ['Scene1', 'Scene7', 'Scene8'],
 ['Scene1', 'Scene7', 'Scene9'],
 ['Scene1', 'Scene8'],
 ['Scene1', 'Scene9'],
 ['Scene2'],
 ['Scene2', 'Scene3'],
 ['Scene2', 'Scene3', 'Scene4'],
 ['Scene2', 'Scene3', 'Scene5'],
 ['Scene2', 'Scene3', 'Scene7'],
 ['Scene2', 'Scene3', 'Scene9'],
 ['Scene2', 'Scene4'],
 ['Scene2', 'Scene4', 'Scene5'],
 ['Scene2', 'Scene4', 'Scene7'],
 ['Scene2', 'Scene5'],
 ['Scene2', 'Scene5', 'Scene7'],
 ['Scene2', 'Scene5', 'Scene9'],
 ['Scene2', 'Scene6'],
 ['Scene2', 'Scene6', 'Scene7'],
 ['Sc

In [11]:
# --- ILP_pos Model Setup --- #

ILP_pos = gp.Model("ILP_pos", env=env)

# decision variables
x_pos = ILP_pos.addVars(n, H, D, vtype=GRB.BINARY, name="x_pos")
y_pos = ILP_pos.addVars(n, n, H-1, D, vtype=GRB.BINARY, name="y_pos")

# aux variable (tau)
tau = ILP_pos.addVars(m, 2, lb=1, ub=D, vtype=GRB.INTEGER, name="tau")

# objective function
objFunc_pos = ILP_pos.setObjective(
    gp.quicksum(
        holdingCost[i] * (tau[i, 1] - tau[i, 0] + 1)
        for i in range(m)
    ), GRB.MINIMIZE
)

# --- constraints (they aint gonna be pretty) o7 --- #
c5 = ILP_pos.addConstrs((
    gp.quicksum (
        x_pos[j, r, d]
        for d in range(D)
        for r in range(H)
    ) == 1
    for j in range(n)),
    name="c5"
) # Constraint 5

c6 = ILP_pos.addConstrs((
    gp.quicksum(
        x_pos[j, r, d]
        for j in range(n)
    ) <= 1
    for d in range(D)
    for r in range(H)),
    name = "c6"
) # Constraint 6

c7 = ILP_pos.addConstrs((
    gp.quicksum(
        x_pos[j, r+1, d]
        for j in range(n)
    ) <=
    gp.quicksum(
        x_pos[j, r, d]
        for j in range(n)
    )
    for d in range(D)
    for r in range(H-1)),
    name="c7"
) # Constraint 7

c8 = ILP_pos.addConstrs(
    (gp.quicksum(
        x_pos[j, 1, d+1]
        for j in range(n)
    ) <=
    gp.quicksum(
        x_pos[j, 1, d]
        for j in range(n)
    ) for d in range(D-1)),
    name="c8"
) # Constraint 8

c9 = ILP_pos.addConstrs((
    tau[i, 0] <= gp.quicksum(
        d * x_pos[j, r, d]
        for r in range(H)
        for d in range(D)
    )for i in range(m)
     for j in range(n)
     if T[i,j]),
    name="c9"
) # Constraint 9

c10 = ILP_pos.addConstrs((
    gp.quicksum(
        d * x_pos[j, r, d]
        for r in range(H)
        for d in range(D)
    ) <= tau[i, 1]
     for i in range(m)
     for j in range(n)
     if T[i,j]),
    name="c10"
) # Constraint 10

c11 = ILP_pos.addConstrs((
    x_pos[j1, r, d] + x_pos[j2, r+1, d] <= y_pos[j1, j2, r, d] + 1
    for d in range(D)
    for j1 in range(n)
    for j2 in range(n)
    for r in range(H-1)
    if j1 != j2),
    name = "c11"
) #Constraint 11

c12 = ILP_pos.addConstrs((
    gp.quicksum(
        y_pos[j1, j2,r,d] + y_pos[j2,j1,r,d]
        for r in range(H-1)
    )<= 1
    for j1 in range(n)
    for j2 in range(n)
    for d in range (D)
    if j1 != j2),
    name = "c12"
) #Constraint 12

c13 = ILP_pos.addConstrs((
    gp.quicksum(
        durations[j] * x_pos[j,r,d]
        for r in range(H)
        for j in range(n)
    ) + gp.quicksum(
        S[j1,j2]*y_pos[j1,j2,r,d]
        for r in range(H-1)
        for j1 in range(n)
        for j2 in range(n)
        if j1 != j2
    )<= W
    for d in range (D)),
    name = "c13"
) #Constraint 13

# --- Constraints (14-16) --- #
# These constraints are embedded in the init of the variables

In [12]:
ILP_pos.optimize()
ILP_pos.getJSONSolution()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: AMD Ryzen 7 4700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Academic license 2773076 - for non-commercial use only - registered to km___@uga.edu
Optimize a model with 2765 rows, 2529 columns and 16038 nonzeros (Min)
Model fingerprint: 0x5877a7c2
Model has 18 linear objective coefficients and an objective constant of 6970
Variable types: 0 continuous, 2529 integer (2511 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [6e+02, 9e+02]
  Bounds range     [1e+00, 9e+00]
  RHS range        [1e+00, 1e+02]

Presolve removed 2664 rows and 2439 columns
Presolve time: 0.02s
Presolved: 101 rows, 90 columns, 816 nonzeros
Variable types: 0 continuous, 90 integer (72 binary)

Root relaxation: infeasible, 107 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    

'{ "SolutionInfo": { "Status": 3, "Runtime": 8.2999944686889648e-02, "Work": 1.8335632480606406e-02, "ObjBound": 1e+100, "ObjBoundC": 1e+100, "MIPGap": 1e+100, "IterCount": 107, "BarIterCount": 0, "NLBarIterCount": 0, "PDHGIterCount": 0, "NodeCount": 1, "SolCount": 0}}'

In [13]:
# ---  Computational Comparison --- #
# Number of scenes 𝑛 ∈ {10, 15, 20, 30, 40, 50, 60};
# Number of actors 𝑚 ∈ {10, 20};
# Capacity 𝑊 ∈ {50, 75};
# Density 𝜌 ∈ {0.3, 0.5} controls the probability for an actor to be present in a scene:
#   if the real number randomly drawn in [0, 1] is smaller than or equal to 𝜌, then 𝑡
#   𝑖,𝑗 = 1; 0, otherwise;
# Duration 𝓁𝑗 was randomly sampled from the integer uniform distribution [1, 20];
# Changeover time 𝑠𝑖,𝑗 was randomly sampled from the integer uniform distribution [1, 10].

scenes = [10, 15, 20, 30, 40, 50, 60]
actors = [10, 20]
W = [50, 75]
p = [0.3, 0.5]
durations = list(randint(1, 21, (1, i)) for i in scenes)
S = list(randint(1, 11, (i, i)) for i in scenes)
T = list((rand(i, j) <= x).astype(int) for i in actors for j in scenes for x in p)
